<h2 style="color:#1E90FF;">
<b> Phase distribution prediction of Computational Fluid Dynamics–Deep Neural Network Model in Slurry Bubble Column Reactors </b>
</h2> 

In [1]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, BatchNormalization, Dropout
import joblib

import pandas as pd
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, BatchNormalization, Dropout
import joblib

In [2]:
import h5py

# First, let's see the EXACT top-level keys
with h5py.File('cfd_weights.weights.h5', 'r') as f:
    print("Top-level keys:")
    for key in f.keys():
        print(repr(key))  # repr() shows exact characters including backslashe

Top-level keys:
'_layer_checkpoint_dependencies\\batch_normalization'
'_layer_checkpoint_dependencies\\batch_normalization_2'
'_layer_checkpoint_dependencies\\batch_normalization_4'
'_layer_checkpoint_dependencies\\dense'
'_layer_checkpoint_dependencies\\dense_2'
'_layer_checkpoint_dependencies\\dense_4'
'_layer_checkpoint_dependencies\\dense_6'
'_layer_checkpoint_dependencies\\dropout'
'_layer_checkpoint_dependencies\\dropout_1'
'_layer_checkpoint_dependencies\\dropout_2'
'metrics\\mean'
'metrics\\mean_metric_wrapper'
'optimizer'
'vars'


In [3]:
import h5py
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, BatchNormalization, Dropout, Input
import joblib

# ── Step 1: Extract weights using FLAT keys ──────────────────────────────────

weights_map = {}

with h5py.File('cfd_weights.weights.h5', 'r') as f:
    layer_names = ['dense', 'batch_normalization', 'dense_2', 
                   'batch_normalization_2', 'dense_4', 
                   'batch_normalization_4', 'dense_6']
    
    for layer_name in layer_names:
        flat_key = f'_layer_checkpoint_dependencies\\{layer_name}'
        group = f[flat_key]
        if 'vars' in group:
            vars_group = group['vars']
            weights_map[layer_name] = [
                vars_group[str(i)][:] 
                for i in range(len(vars_group))
            ]
            print(f"✅ {layer_name}: {[w.shape for w in weights_map[layer_name]]}")

# ── Step 2: Rebuild model ────────────────────────────────────────────────────

model = Sequential([
    Input(shape=(6,)),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.1),
    Dense(2, activation='linear')
])

model.build(input_shape=(None, 6))
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# ── Step 3: Set weights layer by layer ──────────────────────────────────────

layer_order = [
    'dense',                  # Dense(256)
    'batch_normalization',    # BN 1
    'dense_2',                # Dense(128)
    'batch_normalization_2',  # BN 2
    'dense_4',                # Dense(64)
    'batch_normalization_4',  # BN 3
    'dense_6',                # Dense(2) output
]

# Only layers that have weights (skip Dropout)
trainable_layers = [l for l in model.layers if len(l.get_weights()) > 0]

for keras_layer, saved_name in zip(trainable_layers, layer_order):
    w = weights_map[saved_name]
    keras_layer.set_weights(w)
    print(f"✅ Loaded: {keras_layer.name} ← {saved_name} {[x.shape for x in w]}")

# ── Step 4: Load scaler + verify ─────────────────────────────────────────────

scaler = joblib.load('scaler.save')

dummy = np.zeros((1, 6))
dummy_scaled = scaler.transform(dummy)
pred = model.predict(dummy_scaled, verbose=0)

print("\n✅ Model loaded successfully — no retraining needed!")
print("Test prediction:", pred)

✅ dense: [(6, 256), (256,)]
✅ batch_normalization: [(256,), (256,), (256,), (256,)]
✅ dense_2: [(256, 128), (128,)]
✅ batch_normalization_2: [(128,), (128,), (128,), (128,)]
✅ dense_4: [(128, 64), (64,)]
✅ batch_normalization_4: [(64,), (64,), (64,), (64,)]
✅ dense_6: [(64, 2), (2,)]
✅ Loaded: dense ← dense [(6, 256), (256,)]
✅ Loaded: batch_normalization ← batch_normalization [(256,), (256,), (256,), (256,)]
✅ Loaded: dense_1 ← dense_2 [(256, 128), (128,)]
✅ Loaded: batch_normalization_1 ← batch_normalization_2 [(128,), (128,), (128,), (128,)]
✅ Loaded: dense_2 ← dense_4 [(128, 64), (64,)]
✅ Loaded: batch_normalization_2 ← batch_normalization_4 [(64,), (64,), (64,), (64,)]
✅ Loaded: dense_3 ← dense_6 [(64, 2), (2,)]

✅ Model loaded successfully — no retraining needed!
Test prediction: [[0.28242996 0.06842552]]


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.4.1.post1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [6]:
input_names = ["ug(m/s)", "Cv(%)", "P(MPa)", "x(m)", "y(m)", "z(m)"]
output_names = ["Gas Volume Fraction", "Solid Volume Fraction"]

import pandas as pd

# Input feature names
input_names = ["ug(m/s)", "Cv(%)", "P(MPa)", "x(m)", "y(m)", "z(m)"]

# Take user inputs
ug = float(input("Enter ug(m/s): "))
cv = float(input("Enter Cv(%): "))
p = float(input("Enter P(MPa): "))
x = float(input("Enter x(m): "))
y = float(input("Enter y(m): "))
z = float(input("Enter z(m): "))

# Create dataframe for single prediction
input_df = pd.DataFrame({
    "ug(m/s)": [ug],
    "Cv(%)": [cv],
    "P(MPa)": [p],
    "x(m)": [x],
    "y(m)": [y],
    "z(m)": [z]
})

# Scale inputs
X_scaled = scaler.transform(input_df)

# Predict outputs
y_pred = model.predict(X_scaled)

# Output names
output_names = ["Gas Volume Fraction", "Solid Volume Fraction"]

# Print results
print("\n===== Prediction Results =====")
for name, value in zip(output_names, y_pred[0]):
    print(f"{name}: {value:.6f}")

Enter ug(m/s):  0.1
Enter Cv(%):  10
Enter P(MPa):  0.2
Enter x(m):  0
Enter y(m):  0.3
Enter z(m):  0


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step

===== Prediction Results =====
Gas Volume Fraction: 0.264004
Solid Volume Fraction: 0.074532


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


In [8]:
# Step 1: Load Excel file
data_df = pd.read_excel('ug.xlsx')  # Put your actual file name

# Step 2: Scale inputs
X_pred_scaled = scaler.transform(data_df)

# Step 4: Predict
y_pred = model.predict(X_pred_scaled)

# Step 5: Prepare results dataframe with correct output column names
output_columns = ['Pred_Gas_Volume_Fraction', 'Pred_Solids_Volume_Fraction']
pred_df = pd.DataFrame(y_pred, columns=output_columns)

# Step 6: Concatenate with original input data
result_df = pd.concat([data_df.reset_index(drop=True), pred_df.reset_index(drop=True)], axis=1)

# Step 7: Save full results
result_df.to_excel('predicted.xlsx', index=False)
#result_df.to_csv('predicted.csv', index=False)

print("Predictions saved with outputs: gas and solids volume fractions.")

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
Predictions saved with outputs: gas and solids volume fractions.


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
